# GN_DW_POC ETL 파이프라인 — 개선 설계 노트북

> 본 노트북은 기존 `Excel_to_Table.ipynb` 와 그 해설 문서(`Excel_to_Table_해설.md`)의 **비판적 리뷰**를 바탕으로, 더 안전하고 유지보수 가능한 ETL 파이프라인을 **어떻게 재설계할지** 설명합니다.
>
> - **목적**: 단순 코드 나열이 아니라, *문제 → 개선 원칙 → Before/After 패턴* 으로 정리한 설계 가이드
> - **실행 전제**: 코드 셀은 권장 패턴을 보여주는 **예시(템플릿)** 이며, 실제 스테이지/데이터에 맞춰 적용합니다.

## 기존 파이프라인 한 줄 요약
스테이지 Excel → pandas 파싱 → Snowpark DataFrame 적재(RAW 18개) → ANALYTICS 뷰(13개) 구성. 이후 3차례 재적재(교체/추가/신규)가 **하나의 노트북에 순차 누적**되어 있음.

## 개선이 다루는 6개 영역
| # | 영역 | 핵심 개선 |
|---|------|----------|
| 1 | 구조·유지보수성 | 설정 외부화, 공통 유틸 단일화, 재적재 단위 분리 |
| 2 | 데이터 품질·안전성 | 원자적 교체(CREATE OR REPLACE / SWAP), 적재 후 검증, 날짜 타입 통일 |
| 3 | 성능 | COPY INTO, 배치 통합 적재, 임시파일 정리 |
| 4 | 운영·거버넌스 | 전용 ETL Role, 멱등성, 감사 로깅, 환경 분리 |
| 5 | 뷰 설계 | RAW 타입 정착으로 TRY_TO_* 제거, 공통 스테이징, 의존성 문서화 |
| 6 | 권장 아키텍처 | dbt, Task/Stream 자동화, 데이터 계약(Data Contract) |

---
## 1. 구조·유지보수성

| 문제 (기존) | 개선 원칙 |
|------|----------|
| 초기 적재 + 3차 재적재가 하나의 노트북에 누적 → 현재 상태 파악 불가 | 재적재 단위를 별도 노트북/스크립트로 분리하고, 본 노트북은 **선언적 설정**만 둠 |
| `cs()`, `clean_dates()`, `download_from_stage()` 가 섹션마다 재정의 | 공통 유틸을 **단일 모듈/단일 셀**로 정의 후 재사용 |
| 컬럼명이 리터럴 리스트로 하드코딩 → 헤더 변경 시 전수 수정 | 테이블별 스펙을 **TABLE_CONFIGS(설정)** 으로 외부화 |

### 핵심 아이디어: 설정 기반(config-driven) 적재
적재 로직은 한 번만 작성하고, 테이블마다 달라지는 부분(파일명·시트·컬럼·타입)은 **데이터(config)** 로 분리합니다.

In [ ]:
# ============================================================
# [BEFORE] 테이블마다 컬럼/타입/적재 코드를 반복 작성
# - 동일 패턴이 18개 테이블 + 재적재마다 복붙됨
# - Excel 헤더가 바뀌면 모든 셀을 손으로 고쳐야 함
# ============================================================

# 예시 (기존 방식)
fp = download_from_stage('공통_캠페인코드.xlsx')
df = pd.read_excel(fp, sheet_name='공통캠페인코드')
kcols = ['브랜드코드', '브랜드명', '브랜드사용여부', '브랜드사용부서', '상위캠페인코드', '상위캠페인명',
         '상위캠페인사용여부', '세부캠페인코드', '세부캠페인명', '세부캠페인사용부서(실적부서)',
         '세부캠페인사용여부', '세부캠페인시작일']
df.columns = kcols
df = cs(df, kcols)
col_defs = ', '.join([f'"{c}" VARCHAR' for c in kcols])
load_to_table(df, 'DIM_CAMPAIGN_CODE', col_defs)
# ... 이 블록이 테이블 개수만큼 반복 ...

In [ ]:
# ============================================================
# [AFTER] 테이블 스펙을 선언적 설정으로 외부화
# - 컬럼/타입/시트/파일을 한 곳에서 관리
# - 실무에서는 이 dict 를 별도 YAML/JSON 으로 분리 권장 (데이터 계약, 6장 참고)
# ============================================================

from typing import TypedDict, Literal

# 컬럼 타입은 STRING / NUMBER / DATE 3종으로 표준화 (2장 날짜 통일 참고)
TABLE_CONFIGS = {
    'DIM_CAMPAIGN_CODE': {
        'file': '공통_캠페인코드.xlsx',
        'sheet': '공통캠페인코드',
        'load_mode': 'replace',          # replace | append | merge
        'columns': {
            '브랜드코드': 'STRING', '브랜드명': 'STRING', '브랜드사용여부': 'STRING',
            '브랜드사용부서': 'STRING', '상위캠페인코드': 'STRING', '상위캠페인명': 'STRING',
            '상위캠페인사용여부': 'STRING', '세부캠페인코드': 'STRING', '세부캠페인명': 'STRING',
            '세부캠페인사용부서(실적부서)': 'STRING', '세부캠페인사용여부': 'STRING',
            '세부캠페인시작일': 'DATE',
        },
        'primary_key': ['세부캠페인코드'],   # 검증/멱등 MERGE 키 (2장,4장)
    },
    'DIM_MEMBER_ATTRIBUTE': {
        'file': '공통_회원특성.xlsx',
        'sheet': '회원특성',
        'load_mode': 'replace',
        'columns': {'회원번호': 'STRING', '성별': 'STRING', '연령대': 'STRING', '지역': 'STRING'},
        'primary_key': ['회원번호'],
    },
    # ... 나머지 테이블도 동일 구조로 추가 ...
}

SQL_TYPE = {'STRING': 'VARCHAR', 'NUMBER': 'NUMBER', 'DATE': 'DATE'}

print(f'설정된 테이블 수: {len(TABLE_CONFIGS)}')

In [ ]:
# ============================================================
# [AFTER] 공통 유틸을 단 한 번만 정의 → 모든 테이블이 재사용
#   - 중복 정의로 인한 불일치 위험 제거
#   - 적재/검증/로깅을 한 흐름으로 통합 (2,4장에서 확장)
# ============================================================

def build_col_defs(columns: dict) -> str:
    return ', '.join(f'"{c}" {SQL_TYPE[t]}' for c, t in columns.items())

def coerce_types(df, columns: dict):
    """설정의 타입에 따라 일괄 변환 (STRING/NUMBER/DATE)"""
    for col, t in columns.items():
        if t == 'STRING':
            df[col] = df[col].astype('string').replace({'nan': None, 'NaT': None, 'None': None})
        elif t == 'NUMBER':
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif t == 'DATE':
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.date  # → Snowflake DATE
    return df

def load_table(session, name: str, cfg: dict, df):
    cols = cfg['columns']
    df = df[list(cols.keys())].copy()
    df = coerce_types(df, cols)
    fqn = f'GN_DW_POC.RAW.{name}'
    col_defs = build_col_defs(cols)
    # 원자적 교체는 2장 참고. 여기서는 패턴만 표시.
    session.sql(f'CREATE OR REPLACE TABLE {fqn} ({col_defs})').collect()
    session.create_dataframe(df).write.mode('append').save_as_table(fqn)
    return len(df)

# 전체 적재가 단 몇 줄로 단순화됨
# for name, cfg in TABLE_CONFIGS.items():
#     df = pd.read_excel(download(cfg['file']), sheet_name=cfg['sheet'])
#     df.columns = list(cfg['columns'].keys())
#     n = load_table(session, name, cfg, df)
#     print(f'{name}: {n:,} rows')
print('단일 적재 함수 정의 (예시)')

---
## 2. 데이터 품질·안전성

| 문제 (기존) | 개선 원칙 |
|------|----------|
| `DROP → CREATE → INSERT` 교체: 적재 실패 시 기존 데이터 소실 | **원자적 교체**: `CREATE OR REPLACE ... AS SELECT` 또는 임시테이블 적재 후 `SWAP WITH` |
| 적재 후 COUNT만 확인 | NULL 비율·중복 PK·행 수 **assertion 검증** |
| 날짜가 `NUMBER(YYYYMMDD)` / `VARCHAR` / datetime 혼재 | 날짜 컬럼을 **DATE/TIMESTAMP 로 통일** → 뷰에서 TRY_TO_DATE 불필요 |
| 스키마 변경 시 뷰가 즉시 깨짐 | 스키마 **버전·마이그레이션 전략** 도입 |

In [ ]:
# ============================================================
# [개선] 무중단·원자적 교체: 스테이징 테이블에 먼저 적재 → 검증 통과 시 SWAP
#   - 적재 도중 실패해도 운영 테이블(기존 데이터)은 그대로 유지
#   - SWAP 은 메타데이터 연산이라 즉시 완료
# ============================================================

def load_table_safe(session, name: str, cfg: dict, df):
    cols = cfg['columns']
    df = coerce_types(df[list(cols.keys())].copy(), cols)
    target = f'GN_DW_POC.RAW.{name}'
    staging = f'GN_DW_POC.RAW.{name}__STG'
    col_defs = build_col_defs(cols)

    # 1) 스테이징에 적재
    session.sql(f'CREATE OR REPLACE TABLE {staging} ({col_defs})').collect()
    session.create_dataframe(df).write.mode('append').save_as_table(staging)

    # 2) 검증 (실패 시 예외 → 운영 테이블 보존)
    validate_loaded(session, staging, cfg)

    # 3) 원자적 교체
    session.sql(f'CREATE TABLE IF NOT EXISTS {target} ({col_defs})').collect()
    session.sql(f'ALTER TABLE {target} SWAP WITH {staging}').collect()
    session.sql(f'DROP TABLE IF EXISTS {staging}').collect()
    return len(df)

print('원자적 교체 패턴 정의 (예시)')

In [ ]:
# ============================================================
# [개선] 적재 직후 데이터 품질 검증
#   - 행 수 0 방지 / PK 중복 / 필수 컬럼 NULL 비율 임계치
# ============================================================

def validate_loaded(session, fqn: str, cfg: dict, null_threshold: float = 0.5):
    n = session.sql(f'SELECT COUNT(*) C FROM {fqn}').collect()[0]['C']
    assert n > 0, f'[검증실패] {fqn}: 적재된 행이 0건'

    pk = cfg.get('primary_key')
    if pk:
        pk_cols = ', '.join(f'"{c}"' for c in pk)
        dup = session.sql(
            f'SELECT COUNT(*) C FROM (SELECT {pk_cols} FROM {fqn} GROUP BY {pk_cols} HAVING COUNT(*)>1)'
        ).collect()[0]['C']
        assert dup == 0, f'[검증실패] {fqn}: PK 중복 {dup}건'

    for col in cfg['columns']:
        null_ratio = session.sql(
            f'SELECT COUNT_IF("{col}" IS NULL)/NULLIF(COUNT(*),0) R FROM {fqn}'
        ).collect()[0]['R'] or 0
        if null_ratio > null_threshold:
            print(f'  [경고] {fqn}.{col} NULL 비율 {null_ratio:.1%}')
    print(f'  [검증통과] {fqn}: {n:,} rows')

print('검증 함수 정의 (예시)')

### 날짜 타입 통일
- 기존: `신청일`=NUMBER(YYYYMMDD), `중단일`=VARCHAR, GA `날짜`=문자/숫자 혼재
- 개선: 적재 시 `coerce_types(... 'DATE')` 로 **Snowflake DATE** 로 정착
- 효과: 뷰에서 `TRY_TO_DATE(TO_VARCHAR("신청일"),'YYYYMMDD')` 같은 변환이 사라지고, 날짜 범위 필터·DATEDIFF 가 단순해짐

```sql
-- Before (뷰마다 반복)
DATEDIFF('month', TRY_TO_DATE(TO_VARCHAR("신청일"),'YYYYMMDD'), TRY_TO_DATE("중단일"))
-- After (RAW 가 이미 DATE)
DATEDIFF('month', "신청일", "중단일")
```

---
## 3. 성능

| 문제 (기존) | 개선 원칙 |
|------|----------|
| 수십만 행을 `session.create_dataframe(df)` 로 직렬화 적재 → 병목 | Excel → **Parquet** 변환 후 스테이지 업로드 → `COPY INTO` |
| SMS 13개 파일을 반복문에서 13번 `append` 커밋 | 먼저 `pd.concat` 후 **한 번에** 적재 |
| `tempfile.mkdtemp()` 디렉토리 미정리 | `try/finally` + `shutil.rmtree` 로 **정리** |

In [ ]:
# ============================================================
# [개선] 대용량은 Parquet + COPY INTO 가 훨씬 빠르고 안정적
# ============================================================
import shutil, tempfile, os

def load_via_copy(session, name: str, cfg: dict, df):
    cols = cfg['columns']
    df = coerce_types(df[list(cols.keys())].copy(), cols)
    target = f'GN_DW_POC.RAW.{name}'
    tmp = tempfile.mkdtemp()
    try:
        pq = os.path.join(tmp, f'{name}.parquet')
        df.to_parquet(pq, index=False)
        session.file.put(f'file://{pq}', '@GN_DW_POC.RAW.STG_POC_DATA/_parquet/',
                         auto_compress=False, overwrite=True)
        session.sql(f'CREATE OR REPLACE TABLE {target} ({build_col_defs(cols)})').collect()
        session.sql(f'''
            COPY INTO {target}
            FROM @GN_DW_POC.RAW.STG_POC_DATA/_parquet/{name}.parquet
            FILE_FORMAT=(TYPE=PARQUET)
            MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE
        ''').collect()
    finally:
        shutil.rmtree(tmp, ignore_errors=True)   # 임시파일 정리
    return len(df)

print('COPY INTO 적재 패턴 정의 (예시)')

In [ ]:
# ============================================================
# [개선] SMS/알림톡 13개 파일: concat 후 1회 적재
#   - 13번 커밋 → 1번으로 축소, 임시 디렉토리는 finally 에서 정리
# ============================================================

def load_multifile(session, name: str, cfg: dict, files: list, sheet_fn):
    tmp = tempfile.mkdtemp()
    try:
        frames = []
        for f in files:
            local = download_to(session, f, tmp)
            part = pd.read_excel(local, sheet_name=sheet_fn(f))
            part.columns = list(cfg['columns'].keys())
            frames.append(part)
        df = pd.concat(frames, ignore_index=True)   # 한 번에 합침
        return load_table_safe(session, name, cfg, df)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

print('다중 파일 통합 적재 패턴 정의 (예시)')

---
## 4. 운영·거버넌스

| 문제 (기존) | 개선 원칙 |
|------|----------|
| 모든 작업을 `ACCOUNTADMIN` 으로 실행 | 최소 권한 **전용 ETL Role** 생성·사용 |
| 같은 셀 재실행 시 APPEND 중복 | **MERGE** 또는 DELETE+INSERT 로 멱등성 확보 |
| 적재 이력 추적 불가 | **감사 로그 테이블**에 일시·건수·상태 기록 |
| 개발/운영 환경 미분리 | DB/스키마 분리 + 파라미터화 |

In [ ]:
%%sql -r etl_role_setup
-- [개선] ACCOUNTADMIN 대신 전용 ETL 역할 사용 (최소 권한 원칙)
USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS GN_ETL_ROLE;

GRANT USAGE ON DATABASE GN_DW_POC TO ROLE GN_ETL_ROLE;
GRANT USAGE ON SCHEMA GN_DW_POC.RAW TO ROLE GN_ETL_ROLE;
GRANT USAGE ON SCHEMA GN_DW_POC.ANALYTICS TO ROLE GN_ETL_ROLE;
GRANT CREATE TABLE, CREATE VIEW ON SCHEMA GN_DW_POC.RAW TO ROLE GN_ETL_ROLE;
GRANT CREATE VIEW ON SCHEMA GN_DW_POC.ANALYTICS TO ROLE GN_ETL_ROLE;
GRANT READ, WRITE ON STAGE GN_DW_POC.RAW.STG_POC_DATA TO ROLE GN_ETL_ROLE;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE GN_ETL_ROLE;
-- GRANT ROLE GN_ETL_ROLE TO USER <etl_service_user>;

In [ ]:
%%sql -r audit_table_setup
-- [개선] 적재 이력 감사 로그: 일시·테이블·건수·상태·모드 기록
CREATE TABLE IF NOT EXISTS GN_DW_POC.RAW._ETL_AUDIT_LOG (
    RUN_ID        STRING,
    TABLE_NAME    STRING,
    LOAD_MODE     STRING,          -- replace | append | merge
    ROW_COUNT     NUMBER,
    STATUS        STRING,          -- SUCCESS | FAILED
    MESSAGE       STRING,
    LOADED_AT     TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

SELECT * FROM GN_DW_POC.RAW._ETL_AUDIT_LOG ORDER BY LOADED_AT DESC LIMIT 20;

In [ ]:
# ============================================================
# [개선] 멱등성 + 감사 로깅 통합 래퍼
#   - replace : CREATE OR REPLACE / SWAP (재실행해도 동일 결과)
#   - merge   : PK 기준 MERGE (재실행 중복 방지)
#   - 모든 적재 결과를 _ETL_AUDIT_LOG 에 기록
# ============================================================
import uuid

def run_load(session, name: str, cfg: dict, df, run_id=None):
    run_id = run_id or str(uuid.uuid4())[:8]
    try:
        mode = cfg.get('load_mode', 'replace')
        if mode == 'replace':
            n = load_table_safe(session, name, cfg, df)
        elif mode == 'merge':
            n = merge_table(session, name, cfg, df)   # PK 기반 upsert
        else:
            n = append_table(session, name, cfg, df)
        _log_audit(session, run_id, name, mode, n, 'SUCCESS', '')
        return n
    except Exception as e:
        _log_audit(session, run_id, name, cfg.get('load_mode'), 0, 'FAILED', str(e)[:500])
        raise

def _log_audit(session, run_id, name, mode, n, status, msg):
    session.sql(
        "INSERT INTO GN_DW_POC.RAW._ETL_AUDIT_LOG "
        "(RUN_ID, TABLE_NAME, LOAD_MODE, ROW_COUNT, STATUS, MESSAGE) "
        f"VALUES ('{run_id}','{name}','{mode}',{n},'{status}','{msg.replace(chr(39),chr(34))}')"
    ).collect()

print('멱등 적재 + 감사 로깅 래퍼 정의 (예시)')

---
## 5. 뷰 설계

| 문제 (기존) | 개선 원칙 |
|------|----------|
| `TRY_TO_NUMBER`/`TRY_TO_DATE` 남발 (RAW 타입 미지정) | 적재 시 올바른 타입 정착 → 뷰가 단순·고속 |
| `V_CHANNEL_ROI` → `V_MEDIA_EFFICIENCY_DETAIL` 등 뷰 체인 미문서화 | 의존성 다이어그램·계층 명시 |
| 매체별(DRTV/디지털/재송출) 동일 구조를 4개 뷰에서 UNION ALL 반복 | **공통 스테이징 뷰/테이블**로 통합 후 재사용 |

### 뷰 의존성 계층 (권장)
```
RAW (정타입 테이블)
   └─ STG_MEDIA_UNIFIED        ← DRTV+디지털+재송출 UNION 1회만
        ├─ V_MEDIA_EFFICIENCY_DETAIL
        ├─ V_TIME_SLOT_EFFICIENCY
        ├─ V_CHANNEL_ROI        (STG 재사용)
        └─ V_CAMPAIGN_ROI       (STG 재사용)
```

In [ ]:
%%sql -r stg_media_unified
-- [개선] 매체 통합 로직을 공통 스테이징 뷰에 1회만 정의
--        이후 분석 뷰들은 이 뷰를 재사용 → UNION ALL 중복 제거
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.STG_MEDIA_UNIFIED AS
SELECT 'DRTV'   AS "매체구분", "월 구분" AS "월", "월별 목표", "월별 실적",
       "집행 예산", "해당연도" AS "연도"
FROM GN_DW_POC.RAW.FACT_DRTV_MONTHLY_DEV
UNION ALL
SELECT '디지털', "월", "월별 개발목표(건)", "월별 개발실적(건)",
       "월별 집행예산(원)", "연도"
FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV
UNION ALL
SELECT '재송출', "월 구분", "월별 목표", "월별 실적",
       "집행 예산", "연도"
FROM GN_DW_POC.RAW.FACT_RETRANSMIT_MONTHLY_DEV;

-- 분석 뷰는 위 STG 를 재사용 (예시)
-- CREATE OR REPLACE VIEW ...V_MEDIA_EFFICIENCY_DETAIL AS SELECT ... FROM STG_MEDIA_UNIFIED ...

---
## 6. 권장 개선 방향 (목표 아키텍처)

1. **dbt 프로젝트로 전환** — RAW/STG/ANALYTICS 를 dbt model 로 관리하여 의존성·테스트(`not_null`,`unique`,`relationships`)·문서화를 자동화. 뷰 체인이 `ref()` 로 명시되어 깨짐 방지.
2. **COPY INTO 기반 적재** — Excel → Parquet 변환 후 스테이지 적재 (3장). pandas 직렬화 병목 제거.
3. **Task / Stream 자동화** — 수동 노트북 실행 대신 `CREATE TASK` 스케줄 + Stream 으로 증분 처리.
4. **데이터 계약(Data Contract)** — 소스 Excel 컬럼 스펙을 YAML 로 정의(1장 `TABLE_CONFIGS` 외부화)하고, 적재 전 스키마 검증.

### 적용 우선순위 (ROI 기준)
| 우선 | 항목 | 난이도 | 효과 |
|------|------|--------|------|
| 1 | 설정 외부화 + 공통 유틸 단일화 | 낮음 | 유지보수 비용 즉시 절감 |
| 2 | 원자적 교체 + 적재 검증 | 낮음 | 데이터 소실 위험 제거 |
| 3 | 감사 로깅 + 멱등 적재 | 중간 | 운영 추적성·재실행 안전 |
| 4 | RAW 타입 정착 + 공통 스테이징 뷰 | 중간 | 뷰 단순화·성능 |
| 5 | dbt + Task 자동화 | 높음 | 장기 운영 표준화 |

In [ ]:
%%sql -r task_skeleton
-- [참고] 노트북/프로시저를 스케줄 자동화하는 Task 스켈레톤
-- CREATE OR REPLACE TASK GN_DW_POC.RAW.T_DAILY_ETL
--   WAREHOUSE = COMPUTE_WH
--   SCHEDULE  = 'USING CRON 0 6 * * * Asia/Seoul'   -- 매일 06:00 KST
-- AS
--   CALL GN_DW_POC.RAW.SP_RUN_ETL();   -- 적재 로직을 저장 프로시저화
-- ALTER TASK GN_DW_POC.RAW.T_DAILY_ETL RESUME;
SELECT 'Task 자동화는 적재 로직을 SP 로 패키징한 뒤 적용' AS NOTE;